#Guia Detalhado para Implantacao da Nova Infraestrutura

✅ Objetivo

- Este guia explica exatamente o que cada membro do time precisa fazer, - incluindo:

- Como destruir a infra antiga

- Como configurar o novo terraform.tfvars (com exemplos)

- Se o bucket do Terraform State é reutilizado ou criado do zero

- Comandos exatos para rodar em ordem

📌 Passo 1: Verificar e Destruir a Infra Antiga

1.1 - Acessar o diretório da infraestrutura antiga

In [ ]:
cd caminho/da/infra-antiga  # Substitua pelo caminho correto

1.2 - Verificar o que será destruído

In [ ]:
terraform plan -destroy

1.3 - Destruir a infraestrutura antiga

In [ ]:
terraform destroy -auto-approve

#📌 Passo 3: Configurar o terraform.tfvars

3.1 - Criar/editar o arquivo terraform.tfvars


In [ ]:
aws_region   = "us-east-1"
environment  = "dev"
owner_tag    = "seu-user"

common_tags = {
  Project     = "consultor-juridico",
  Environment = "dev",
  Owner       = "seu-user",
  CostCenter  = "TI"
}

# Opcional (caso use uma role específica)
aws_assume_role_arn = "arn:aws:iam::619071337533:role/chatbot-role-dev"

### 🔹 O que cada um precisa mudar?

| 🧩 Variável              | ❗ Obrigatório? | 🧪 Exemplo       | 💬 Observação                                |
|--------------------------|----------------|------------------|----------------------------------------------|
| `owner_tag`              | ✅ SIM         | `"joao"`         | 🔑 Deve ser único por pessoa                 |
| `common_tags.Owner`      | ✅ SIM         | 🔁 Igual a cima  | 🔗 Deve bater com `owner_tag`               |
| `environment`            | ⚠️ Não         | `"dev"`          | 🛠️ Padrão `"dev"`, mudar se combinado        |
| `aws_region`             | ⚠️ Não         | `"us-east-1"`    | 🌍 Mudar somente se alinhado com o time      |
| `aws_assume_role_arn`    | ⚠️ Não         | 🆔 ARN da Role   | 🧾 Usar apenas se houver role específica     |


## 📌 Passo 4: Inicializar o Terraform e Verificar o State

### 4.1 - Entendendo o Backend Remoto (Terraform State)

O Terraform precisa armazenar o **estado da infraestrutura provisionada**.  
No nosso projeto, usamos **backend remoto no S3**, o que permite:

- 🔐 Compartilhamento seguro do estado entre membros do time  
- 📜 Histórico de alterações (versionamento automático via S3)  
- 🔒 Bloqueio de concorrência (lock via DynamoDB, se configurado)

---

### ✅ Como está configurado no projeto?

O backend é definido nos arquivos `main.tf` da **raiz da infraestrutura**.

#### 📎 Exemplo simplificado do bloco `backend`:

```hcl
terraform {
  backend "s3" {
    bucket         = "nome-do-bucket"
    key            = "caminho/do/terraform.tfstate"
    region         = "us-east-1"
    dynamodb_table = "nome-da-tabela-lock"
  }
}


#📌 Passo 4: Inicializar o Terraform e Verificar o State

## 4.1 - Configuração do Backend

A configuração do backend remoto está definida no arquivo providers.tf, com o seguinte conteúdo:

In [ ]:
provider "aws" {
  region  = "us-east-1"
  profile = "seu-perfil-aws"   # Altere para o seu profile da AWS CLI

  default_tags {
    tags = var.common_tags
  }
}

terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }

  backend "s3" {
    bucket         = "chatbot-terraform-state-global"       # Bucket já existente e compartilhado
    key            = "chatbot-juridico/terraform.tfstate"   # Caminho fixo no bucket
    region         = "us-east-1"
    encrypt        = true
    use_lockfile   = true                                   # Lock sem DynamoDB
    profile        = "seu-perfil-aws"                       # Alterar para seu profile
  }
}


### ✅ Observações importantes

- ❌ **O bucket não é criado automaticamente** — ele já existe.
- ✅ O bucket usado é **`chatbot-terraform-state-global`**.
- 🔑 A chave (key) usada para o state é **`chatbot-juridico/terraform.tfstate`**, comum ao projeto (não personalizada por integrante).
- 👥 Cada membro do time deve ajustar a linha **`profile`** para o nome do seu perfil da AWS CLI. Certifique-se de usar o perfil correto para que as credenciais sejam usadas corretamente.


# 4.2 - Inicializar o Terraform

In [ ]:
terraform init

### Esse comando irá:

- ☁️ Configurar o backend remoto no S3
- 📥 Baixar os providers necessários
- ✅ Validar a sintaxe e estrutura do projeto

---

### 4.3 - Verificar no Console AWS (opcional)

Você pode verificar o arquivo do state no S3:

1. 🌐 Acesse o **S3 Console AWS**
2. 🗃️ Encontre o bucket **`chatbot-terraform-state-global`**
3. 📂 Navegue até **`chatbot-juridico/terraform.tfstate`**


# 📌 Passo 5: Verificar o Plano de Execução

In [ ]:
terraform plan

🔹 O que verificar?

- Todos os recursos estão sendo criados corretamente?

- owner_tag e common_tags.Owner estão corretos?

- O nome do bucket segue o padrão esperado?

# 📌 Passo 6: Aplicar a Infraestrutura

In [ ]:
terraform apply -auto-approve

🔹 Saída esperada:

instance_public_ip → IP da instância EC2 (não usado com SSM)

docs_bucket_name → Nome do bucket S3

dashboard_url → URL do CloudWatch Dashboard

## 📌 Passo 7: Pós-Implantação — Verificações

### 7.1 - Testar acesso à instância EC2 via AWS SSM

Certifique-se de que a instância tem a **role correta** e o **agente SSM ativado**.

1. 🌐 Acesse o **Console AWS**
2. 🔧 Acesse **AWS Systems Manager > Session Manager**
3. ▶️ Clique em **Start session**
4. 🖥️ Selecione a instância com seu **owner_tag**

---

### ✅ Se a sessão abrir, está funcionando.

---

### 7.2 - Verificar arquivos no bucket S3


In [ ]:
aws s3 ls $(terraform output -raw docs_bucket_name)

#7.3 - Acessar o Dashboard CloudWatch

In [ ]:
terraform output -raw dashboard_url

## 🔹 Resumo Final (Checklist por Integrante)

| 📋 **Passo** | 🛠️ **Ação**                | 💻 **Comando**           | 📝 **Observação**                              |
|--------------|----------------------------|--------------------------|------------------------------------------------|
| 1            | ❌ Destruir infra antiga    | `terraform destroy`      | Apenas se já tiver algo rodando               |
| 2            | 🤖 Clonar repositório       | `git clone ...`          | Todos devem fazer                              |
| 3            | ✏️ Editar `terraform.tfvars`| `nano terraform.tfvars`  | Ajustar `owner_tag` e `common_tags`           |
| 4            | ⚙️ Inicializar Terraform    | `terraform init`         | Usa bucket de state compartilhado             |
| 5            | 📝 Verificar plano          | `terraform plan`         | Validar recursos e nomes                       |
| 6            | 🚀 Aplicar infraestrutura   | `terraform apply`        | Cria recursos                                  |
| 7            | 🔍 Verificações pós-deploy  | Console e comandos AWS   | SSM, S3 e Dashboard CloudWatch                 |

---

### ❓ Dúvidas Comuns

1. **O bucket de Terraform State é compartilhado?**  
   ✅ **Sim**, mas a key é fixa e definida no `providers.tf`, não sendo personalizada por integrante.

2. **E se dois usarem a mesma configuração?**  
   ⚠️ **Usarão o mesmo state file!** Isso pode causar conflitos. Se necessário, altere a key manualmente.

3. **Posso mudar a região?**  
   ⚠️ **Só se todo o time estiver de acordo.**

4. **E se eu esquecer de mudar o `owner_tag`?**  
   ❌ **Risco de sobrepor os recursos de outro membro do time!**
